# 04_tabular_baselines

## Зачем нужен этот ноутбук

Здесь собраны глобальные табличные baseline-модели:
- Ridge;
- Lasso;
- Decision Tree;
- Random Forest.

Ключевое требование: все модели должны обучаться и оцениваться
по одному и тому же воспроизводимому протоколу.

## Что читает

- `artifacts/datasets/train_processed.csv`
- `artifacts/datasets/test_processed.csv`
- `artifacts/datasets/top_pairs.csv`
- `artifacts/metadata/splits.json`

## Что сохраняет

- `artifacts/metrics/<model>_metrics_all_series.csv`
- `artifacts/metrics/<model>_metrics_top_pairs.csv`
- `artifacts/predictions/<model>_preds_<scope>.pkl`
- `artifacts/submissions/submission_<model>.csv`


In [ ]:
from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

PROJECT_ROOT = Path('..')
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
DATASETS_DIR = ARTIFACTS_DIR / 'datasets'
METADATA_DIR = ARTIFACTS_DIR / 'metadata'
METRICS_DIR = ARTIFACTS_DIR / 'metrics'
PREDICTIONS_DIR = ARTIFACTS_DIR / 'predictions'
SUBMISSIONS_DIR = ARTIFACTS_DIR / 'submissions'

for path in [METRICS_DIR, PREDICTIONS_DIR, SUBMISSIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

train_processed = pd.read_csv(DATASETS_DIR / 'train_processed.csv', parse_dates=['date'])
test_processed = pd.read_csv(DATASETS_DIR / 'test_processed.csv', parse_dates=['date'])
top_pairs_df = pd.read_csv(DATASETS_DIR / 'top_pairs.csv')
with open(METADATA_DIR / 'splits.json', 'r', encoding='utf-8') as f:
    splits_json = json.load(f)

splits = []
for s in splits_json:
    splits.append({
        'name': s['name'],
        'train_idx': pd.Index(s['train_idx']),
        'val_idx': pd.Index(s['val_idx']),
        'train_end': pd.to_datetime(s['train_end']),
        'val_start': pd.to_datetime(s['val_start']),
        'val_end': pd.to_datetime(s['val_end']),
    })

top_pairs = set(zip(top_pairs_df['store_nbr'], top_pairs_df['family']))


## Признаковая политика

Для этих моделей сознательно используется безопасный набор признаков:
- `sales_sum` исключен как потенциальная утечка;
- в базовом submission-режиме разрешены только признаки, которые воспроизводимо
  доступны на тестовом горизонте;
- лаговые и rolling-признаки строятся одинаково для всех моделей.


In [ ]:
SAFE_NUMERIC_FEATURES = ['onpromotion']
OPTIONAL_RESEARCH_NUMERIC = [
    col for col in ['dcoilwtico']
    if col in train_processed.columns and col in test_processed.columns
]
NUMERIC_FEATURES = SAFE_NUMERIC_FEATURES + OPTIONAL_RESEARCH_NUMERIC
CATEGORY_FEATURES = ['store_nbr', 'family', 'city', 'state', 'store_type', 'cluster', 'type', 'locale', 'agg_description']
BINARY_FEATURES = ['is_holiday', 'is_payday', 'transferred']
TIME_FEATURES = ['dow', 'weekofyear', 'month', 'day', 'is_weekend']


In [ ]:
def add_time_features(df):
    df = df.copy()
    df['dow'] = df['date'].dt.weekday.astype('int8')
    df['weekofyear'] = df['date'].dt.isocalendar().week.astype('int16')
    df['month'] = df['date'].dt.month.astype('int8')
    df['day'] = df['date'].dt.day.astype('int8')
    df['is_weekend'] = df['dow'].isin([5, 6]).astype('int8')
    return df


def add_lag_features(df, lags=(1, 7, 14, 28)):
    df = df.sort_values(['store_nbr', 'family', 'date']).copy()
    for lag in lags:
        df[f'sales_lag_{lag}'] = df.groupby(['store_nbr', 'family'])['sales'].shift(lag)
    return df


def add_rolling_features(df, windows=(7, 28)):
    df = df.sort_values(['store_nbr', 'family', 'date']).copy()
    grouped = df.groupby(['store_nbr', 'family'])['sales']
    for window in windows:
        df[f'sales_rolling_mean_{window}'] = grouped.shift(1).rolling(window=window, min_periods=1).mean()
    return df


def filter_to_top_pairs(df):
    mask = df[['store_nbr', 'family']].apply(tuple, axis=1).isin(top_pairs)
    return df[mask].copy()


## Общий протокол: обучение, recursive CV и финальный submission

Здесь важно, что итоговая оценка строится рекурсивно на сплитах,
а не только через one-step фичи. Это делает сравнение с CatBoost и SARIMA
методологически более аккуратным.


In [ ]:
def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    return {
        'RMSLE': np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2)),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'WAPE': np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)),
    }


def get_feature_cols(df):
    lag_roll_cols = [c for c in df.columns if c.startswith('sales_lag_') or c.startswith('sales_rolling_mean_')]
    feature_cols = NUMERIC_FEATURES + BINARY_FEATURES + TIME_FEATURES + lag_roll_cols + CATEGORY_FEATURES
    return [c for c in feature_cols if c in df.columns]


def make_pipeline(model):
    numeric_cols = NUMERIC_FEATURES + TIME_FEATURES
    lag_roll_selector = lambda cols: [c for c in cols if c.startswith('sales_lag_') or c.startswith('sales_rolling_mean_')]

    def build_for(feature_cols):
        lag_roll_cols = lag_roll_selector(feature_cols)
        num_cols = [c for c in numeric_cols + lag_roll_cols if c in feature_cols]
        bin_cols = [c for c in BINARY_FEATURES if c in feature_cols]
        cat_cols = [c for c in CATEGORY_FEATURES if c in feature_cols]
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', Pipeline([
                    ('imputer', SimpleImputer(strategy='constant', fill_value=0.0)),
                    ('scaler', StandardScaler()),
                ]), num_cols),
                ('bin', Pipeline([
                    ('imputer', SimpleImputer(strategy='most_frequent')),
                ]), bin_cols),
                ('cat', Pipeline([
                    ('imputer', SimpleImputer(strategy='constant', fill_value='__MISSING__')),
                    ('ohe', OneHotEncoder(handle_unknown='ignore')),
                ]), cat_cols),
            ]
        )
        return Pipeline([('prep', preprocessor), ('model', model)])

    return build_for


MODEL_REGISTRY = {
    'ridge': Ridge(alpha=1.0),
    'lasso': Lasso(alpha=0.0005, max_iter=5000),
    'decision_tree': DecisionTreeRegressor(max_depth=12, random_state=42),
    'random_forest': RandomForestRegressor(n_estimators=150, max_depth=14, random_state=42, n_jobs=-1),
}


In [ ]:
def make_train_dataset(train_df, lags=(1, 7, 14, 28), windows=(7, 28)):
    df = train_df.sort_values(['store_nbr', 'family', 'date']).copy()
    df = add_time_features(df)
    df = add_lag_features(df, lags=lags)
    df = add_rolling_features(df, windows=windows)
    lag_roll_cols = [c for c in df.columns if c.startswith('sales_lag_') or c.startswith('sales_rolling_mean_')]
    df = df.dropna(subset=lag_roll_cols)
    feature_cols = get_feature_cols(df)
    return df, feature_cols


def fit_model_on_train(train_df, model_name):
    train_fe, feature_cols = make_train_dataset(train_df)
    pipeline_builder = make_pipeline(MODEL_REGISTRY[model_name])
    pipeline = pipeline_builder(feature_cols)
    pipeline.fit(train_fe[feature_cols], train_fe['sales'])
    return pipeline, feature_cols


def recursive_forecast_on_frame(model, feature_cols, history_df, future_df):
    hist = history_df.sort_values(['store_nbr', 'family', 'date']).copy()
    future = future_df.sort_values(['date', 'store_nbr', 'family']).copy()
    preds_frames = []

    for cur_date in sorted(future['date'].unique()):
        day_future = future[future['date'] == cur_date].copy()
        day_future['sales'] = np.nan
        combined = pd.concat([hist, day_future], axis=0, ignore_index=True)
        combined = add_time_features(combined)
        combined = add_lag_features(combined)
        combined = add_rolling_features(combined)
        cur_df = combined[combined['date'] == cur_date].copy()
        X_cur = cur_df[feature_cols].copy()
        y_pred = np.clip(model.predict(X_cur), 0, None)

        out = cur_df[['date', 'store_nbr', 'family']].copy()
        out['y_pred'] = y_pred
        preds_frames.append(out)

        hist_update = future[future['date'] == cur_date].copy()
        hist_update['sales'] = y_pred
        hist = pd.concat([hist, hist_update], axis=0, ignore_index=True)

    return pd.concat(preds_frames, ignore_index=True)


In [ ]:
def evaluate_model_recursive(model_name, scope='all_series'):
    use_top_pairs = scope == 'top_pairs'
    metrics_rows = []
    preds_by_split = {}

    for split in splits:
        train_df = train_processed.loc[split['train_idx']].copy()
        val_df = train_processed.loc[split['val_idx']].copy()
        if use_top_pairs:
            train_df = filter_to_top_pairs(train_df)
            val_df = filter_to_top_pairs(val_df)

        model, feature_cols = fit_model_on_train(train_df, model_name)
        preds_df = recursive_forecast_on_frame(model, feature_cols, train_df, val_df.drop(columns=['sales']))
        val_true = val_df[['date', 'store_nbr', 'family', 'sales']].rename(columns={'sales': 'y_true'})
        val_pred = preds_df.merge(val_true, on=['date', 'store_nbr', 'family'], how='left')
        metrics = calculate_metrics(val_pred['y_true'].to_numpy(), val_pred['y_pred'].to_numpy())
        metrics.update({'split': split['name'], 'model': model_name, 'scope': scope})
        metrics_rows.append(metrics)
        preds_by_split[split['name']] = val_pred

    metrics_df = pd.DataFrame(metrics_rows)
    metrics_df.to_csv(METRICS_DIR / f'{model_name}_metrics_{scope}.csv', index=False)
    with open(PREDICTIONS_DIR / f'{model_name}_preds_{scope}.pkl', 'wb') as f:
        pickle.dump(preds_by_split, f)
    return metrics_df, preds_by_split


In [ ]:
def fit_full_model_and_submit(model_name):
    model, feature_cols = fit_model_on_train(train_processed, model_name)
    submission_preds = recursive_forecast_on_frame(
        model,
        feature_cols,
        history_df=train_processed,
        future_df=test_processed.drop(columns=[]),
    )
    submission = (
        test_processed[['id', 'store_nbr', 'family', 'date']]
        .merge(submission_preds, on=['date', 'store_nbr', 'family'], how='left')
        [['id', 'y_pred']]
        .rename(columns={'y_pred': 'sales'})
        .sort_values('id')
    )
    submission.to_csv(SUBMISSIONS_DIR / f'submission_{model_name}.csv', index=False)
    return submission


## Примеры запуска

Ниже оставлены воспроизводимые команды запуска отдельных моделей.
Они не исполняются автоматически, чтобы ноутбук оставался управляемым по времени.


In [ ]:
# ridge_metrics_all, ridge_preds_all = evaluate_model_recursive('ridge', scope='all_series')
# ridge_metrics_top, ridge_preds_top = evaluate_model_recursive('ridge', scope='top_pairs')
# ridge_submission = fit_full_model_and_submit('ridge')
